# 05 — Final Load Preparation
**Project:** FIFA World Cup Analytics  
**Notebook:** `05_final_load_prep.ipynb`

## Purpose
This is the **last Python notebook** in the pipeline. It takes the clean master dataset
from `02_cleaning.ipynb` and produces four Tableau-ready export files — each pre-aggregated
at the correct grain so Tableau needs zero calculated fields to build the dashboard.

## What this notebook does
1. Load the clean master dataset
2. Compute **KPI Table 1 — Tournament Summary** (1 row per edition)
3. Compute **KPI Table 2 — Team Performance** (1 row per team, all-time)
4. Compute **KPI Table 3 — Player Stats** (1 row per player, all-time)
5. Compute **KPI Table 4 — Match Analytics** (1 row per match)
6. Validate all four exports
7. Export to `data/processed/` — ready for Tableau ingestion

## Output Files
| File | Grain | Rows | Purpose |
|------|-------|------|---------|
| `kpi_tournament.csv` | 1 row per edition | 20 | Tournament trends dashboard |
| `kpi_team.csv` | 1 row per team | 83 | Team performance dashboard |
| `kpi_player.csv` | 1 row per player | 7,638 | Player stats dashboard |
| `kpi_match.csv` | 1 row per match | 836 | Match analytics dashboard |

> **Rule:** This notebook does not do any analysis or visualisation. It only computes,
> validates, and exports. All outputs feed directly into Tableau.

---
## 1. Setup & Load

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

# Load the single clean master file produced by 02_cleaning.ipynb
df = pd.read_csv('../data/processed/world_cup_master.csv', parse_dates=['datetime'])

# Match-level view — one row per match
matches = df.drop_duplicates('matchid').copy()
matches['ht_goals']          = matches['half_time_home_goals'] + matches['half_time_away_goals']
matches['second_half_goals'] = matches['total_goals'] - matches['ht_goals']

print(f'Master dataset loaded  : {df.shape}')
print(f'Unique matches         : {len(matches)}')
print(f'Unique players         : {df["player_name"].nunique()}')
print(f'Unique teams           : {df["team_initials"].nunique()}')
print(f'Tournament editions    : {df["year"].nunique()}')
print(f'Year range             : {df["year"].min()} – {df["year"].max()}')

---
## 2. KPI Table 1 — Tournament Summary

**Grain:** One row per World Cup edition (20 rows total)

**KPIs computed:**
- Goals scored, matches played, qualified teams, host country, winner
- Average goals per match
- Average attendance per match
- Whether the host nation won that edition
- Goals per team (scoring efficiency per qualified nation)

In [ ]:
# Base: one row per year from the WorldCups data (already in master)
kpi_tournament = (
    df.drop_duplicates('year')[[
        'year','host_country','winner','runners_up','third','fourth',
        'goalsscored','qualifiedteams','matchesplayed','tournament_attendance'
    ]]
    .sort_values('year')
    .reset_index(drop=True)
)

# KPI: Average goals per match
kpi_tournament['avg_goals_per_match'] = (
    kpi_tournament['goalsscored'] / kpi_tournament['matchesplayed']
).round(2)

# KPI: Average attendance per match
kpi_tournament['avg_attendance_per_match'] = (
    kpi_tournament['tournament_attendance'] / kpi_tournament['matchesplayed']
).round(0).astype(int)

# KPI: Did the host nation win? (1 = Yes, 0 = No)
kpi_tournament['host_nation_won'] = (
    kpi_tournament['host_country'] == kpi_tournament['winner']
).astype(int)

# KPI: Goals per qualified team (how open was the tournament?)
kpi_tournament['goals_per_team'] = (
    kpi_tournament['goalsscored'] / kpi_tournament['qualifiedteams']
).round(2)

# KPI: Era label (for Tableau colour segmentation)
def era_label(year):
    if year <= 1978: return 'Classic Era (≤16 teams)'
    elif year <= 1994: return 'Expansion Era (24 teams)'
    else: return 'Modern Era (32 teams)'

kpi_tournament['era'] = kpi_tournament['year'].apply(era_label)

print(f'kpi_tournament shape: {kpi_tournament.shape}')
print(f'Columns: {list(kpi_tournament.columns)}')
print()
print(kpi_tournament.to_string(index=False))

---
## 3. KPI Table 2 — Team Performance

**Grain:** One row per team, aggregated across all World Cups they participated in

**KPIs computed:**
- Matches played, wins, draws, losses
- Goals scored, goals conceded, goal difference
- Win rate %, goals per match
- Tournament appearances
- Best result (titles won)

In [ ]:
# --- Goals scored (home perspective + away perspective) ---
home_scored   = matches[['home_team_name','home_team_goals','matchid']]\
    .rename(columns={'home_team_name':'team','home_team_goals':'goals_scored'})
away_scored   = matches[['away_team_name','away_team_goals','matchid']]\
    .rename(columns={'away_team_name':'team','away_team_goals':'goals_scored'})

home_conceded = matches[['home_team_name','away_team_goals']]\
    .rename(columns={'home_team_name':'team','away_team_goals':'goals_conceded'})
away_conceded = matches[['away_team_name','home_team_goals']]\
    .rename(columns={'away_team_name':'team','home_team_goals':'goals_conceded'})

all_goals_scored   = pd.concat([home_scored,   away_scored])
all_goals_conceded = pd.concat([home_conceded, away_conceded])

# --- Match results per team ---
home_results = matches[['home_team_name','home_team_goals','away_team_goals']].copy()
home_results['team']   = home_results['home_team_name']
home_results['result'] = home_results.apply(
    lambda r: 'Win' if r['home_team_goals'] > r['away_team_goals']
    else ('Draw' if r['home_team_goals'] == r['away_team_goals'] else 'Loss'), axis=1
)

away_results = matches[['away_team_name','home_team_goals','away_team_goals']].copy()
away_results['team']   = away_results['away_team_name']
away_results['result'] = away_results.apply(
    lambda r: 'Win' if r['away_team_goals'] > r['home_team_goals']
    else ('Draw' if r['away_team_goals'] == r['home_team_goals'] else 'Loss'), axis=1
)

all_results = pd.concat([home_results[['team','result']], away_results[['team','result']]])

# --- Aggregate into team KPI table ---
kpi_team = pd.DataFrame({
    'matches_played'  : all_goals_scored.groupby('team')['matchid'].nunique(),
    'wins'            : all_results[all_results['result']=='Win'].groupby('team').size(),
    'draws'           : all_results[all_results['result']=='Draw'].groupby('team').size(),
    'losses'          : all_results[all_results['result']=='Loss'].groupby('team').size(),
    'goals_scored'    : all_goals_scored.groupby('team')['goals_scored'].sum(),
    'goals_conceded'  : all_goals_conceded.groupby('team')['goals_conceded'].sum(),
}).fillna(0).astype(int)

kpi_team['win_rate_pct']    = (kpi_team['wins'] / kpi_team['matches_played'] * 100).round(1)
kpi_team['goal_difference'] = kpi_team['goals_scored'] - kpi_team['goals_conceded']
kpi_team['goals_per_match'] = (kpi_team['goals_scored'] / kpi_team['matches_played']).round(2)

# Tournament appearances per team
team_years = pd.concat([
    matches[['home_team_name','year']].rename(columns={'home_team_name':'team'}),
    matches[['away_team_name','year']].rename(columns={'away_team_name':'team'})
]).drop_duplicates()
kpi_team['tournament_appearances'] = team_years.groupby('team')['year'].nunique()

# World Cup titles
title_counts = kpi_tournament['winner'].value_counts()
kpi_team['titles_won'] = kpi_team.index.map(title_counts).fillna(0).astype(int)

kpi_team = kpi_team.reset_index().rename(columns={'index':'team'})
kpi_team = kpi_team.sort_values('wins', ascending=False).reset_index(drop=True)

print(f'kpi_team shape: {kpi_team.shape}')
print(f'Columns: {list(kpi_team.columns)}')
print()
print(kpi_team.head(15).to_string(index=False))

---
## 4. KPI Table 3 — Player Stats

**Grain:** One row per player, aggregated across all their World Cup appearances

**KPIs computed:**
- Total appearances (unique matches)
- Goals, yellow cards, red cards, substitutions
- Goals per appearance ratio
- Tournaments attended
- Starting XI appearances vs substitute appearances

In [ ]:
# Parse event codes — each player row has one event field
# G40' = goal at 40 min | Y65' = yellow card | R78' = red card | SU60' = subbed in

df['goal_count']    = df['event'].str.count(r'G\d+')
df['yellow_card']   = df['event'].str.contains(r'Y\d',    regex=True, na=False).astype(int)
df['red_card']      = df['event'].str.contains(r'R\d',    regex=True, na=False).astype(int)
df['subbed_in']     = df['event'].str.contains(r'SU\d|SI\d', regex=True, na=False).astype(int)

# Starting vs Substitute count
starts = df[df['line_up'] == 'Starting'].groupby('player_name')['matchid'].nunique().rename('starts')
subs   = df[df['line_up'] == 'Substitute'].groupby('player_name')['matchid'].nunique().rename('sub_appearances')

kpi_player = df.groupby('player_name').agg(
    team                  =('team_initials', 'first'),
    appearances           =('matchid',       'nunique'),
    goals                 =('goal_count',    'sum'),
    yellow_cards          =('yellow_card',   'sum'),
    red_cards             =('red_card',      'sum'),
    subbed_in_count       =('subbed_in',     'sum'),
    tournaments_attended  =('year',          'nunique'),
    first_year            =('year',          'min'),
    last_year             =('year',          'max'),
).reset_index()

# Merge starting/sub counts
kpi_player = kpi_player.merge(starts.reset_index(), on='player_name', how='left')
kpi_player = kpi_player.merge(subs.reset_index(),   on='player_name', how='left')
kpi_player[['starts','sub_appearances']] = kpi_player[['starts','sub_appearances']].fillna(0).astype(int)

# Derived KPIs
kpi_player['goals_per_appearance'] = (
    kpi_player['goals'] / kpi_player['appearances'].replace(0, np.nan)
).round(3).fillna(0)

kpi_player['career_span_years'] = kpi_player['last_year'] - kpi_player['first_year']

# Sort by goals descending
kpi_player = kpi_player.sort_values('goals', ascending=False).reset_index(drop=True)

print(f'kpi_player shape: {kpi_player.shape}')
print(f'Columns: {list(kpi_player.columns)}')
print()
print(kpi_player.head(15).to_string(index=False))

---
## 5. KPI Table 4 — Match Analytics

**Grain:** One row per match (836 rows)

**KPIs computed:**
- All match facts: teams, goals, stage, stadium, city, year, datetime
- Half-time goals and second-half goals
- Win type (Normal / Extra Time / Penalties) — standardised
- Boolean flags: home win, draw, high-scoring, knockout stage
- Goal margin (winning margin)

In [ ]:
KNOCKOUT_STAGES = [
    'Final', 'Match for third place', 'Semi-finals',
    'Quarter-finals', 'Round of 16'
]

kpi_match = matches[[
    'matchid','year','datetime','stage','stadium','city',
    'home_team_name','away_team_name',
    'home_team_initials','away_team_initials',
    'home_team_goals','away_team_goals',
    'half_time_home_goals','half_time_away_goals',
    'total_goals','win_conditions','attendance',
    'host_country','winner','era'
]].copy() if 'era' in matches.columns else matches[[
    'matchid','year','datetime','stage','stadium','city',
    'home_team_name','away_team_name',
    'home_team_initials','away_team_initials',
    'home_team_goals','away_team_goals',
    'half_time_home_goals','half_time_away_goals',
    'total_goals','win_conditions','attendance',
    'host_country','winner'
]].copy()

# Add era label
def era_label(year):
    if year <= 1978: return 'Classic Era (≤16 teams)'
    elif year <= 1994: return 'Expansion Era (24 teams)'
    else: return 'Modern Era (32 teams)'

kpi_match['era'] = kpi_match['year'].apply(era_label)

# Half-time and second-half goals
kpi_match['ht_goals']          = kpi_match['half_time_home_goals'] + kpi_match['half_time_away_goals']
kpi_match['second_half_goals'] = kpi_match['total_goals'] - kpi_match['ht_goals']

# Winning team and margin
kpi_match['winning_team'] = kpi_match.apply(
    lambda r: r['home_team_name'] if r['home_team_goals'] > r['away_team_goals']
    else (r['away_team_name'] if r['away_team_goals'] > r['home_team_goals'] else 'Draw'), axis=1
)
kpi_match['losing_team'] = kpi_match.apply(
    lambda r: r['away_team_name'] if r['home_team_goals'] > r['away_team_goals']
    else (r['home_team_name'] if r['away_team_goals'] > r['home_team_goals'] else 'Draw'), axis=1
)
kpi_match['goal_margin'] = abs(kpi_match['home_team_goals'] - kpi_match['away_team_goals'])

# Standardise win condition to 3 clean categories
def clean_win_cond(wc):
    wc_lower = wc.lower()
    if 'penalties' in wc_lower: return 'Penalties'
    if 'extra time' in wc_lower or 'golden goal' in wc_lower: return 'Extra Time'
    return 'Normal'

kpi_match['win_type'] = kpi_match['win_conditions'].apply(clean_win_cond)

# Boolean flags (0/1) for Tableau filters
kpi_match['is_home_win']    = (kpi_match['home_team_goals'] > kpi_match['away_team_goals']).astype(int)
kpi_match['is_away_win']    = (kpi_match['away_team_goals'] > kpi_match['home_team_goals']).astype(int)
kpi_match['is_draw']        = (kpi_match['home_team_goals'] == kpi_match['away_team_goals']).astype(int)
kpi_match['is_high_scoring']= (kpi_match['total_goals'] >= 4).astype(int)  # 4+ goals = high-scoring
kpi_match['is_goalless']    = (kpi_match['total_goals'] == 0).astype(int)
kpi_match['is_knockout']    = kpi_match['stage'].isin(KNOCKOUT_STAGES).astype(int)
kpi_match['host_played']    = (
    (kpi_match['home_team_name'] == kpi_match['host_country']) |
    (kpi_match['away_team_name'] == kpi_match['host_country'])
).astype(int)

# Month of match (for seasonal analysis)
kpi_match['match_month'] = kpi_match['datetime'].dt.month
kpi_match['match_day']   = kpi_match['datetime'].dt.day_name()

kpi_match = kpi_match.sort_values(['year','matchid']).reset_index(drop=True)

print(f'kpi_match shape: {kpi_match.shape}')
print(f'Columns ({len(kpi_match.columns)}): {list(kpi_match.columns)}')
print()
print('Quick counts:')
print(f'  High-scoring matches (4+ goals): {kpi_match["is_high_scoring"].sum()}')
print(f'  Goalless draws                 : {kpi_match["is_goalless"].sum()}')
print(f'  Decided by Penalties           : {(kpi_match["win_type"]=="Penalties").sum()}')
print(f'  Decided by Extra Time          : {(kpi_match["win_type"]=="Extra Time").sum()}')
print(f'  Knockout matches               : {kpi_match["is_knockout"].sum()}')

---
## 6. Validation — Check Every KPI Table

Before exporting, verify every table is clean — no unexpected nulls,
no broken KPIs, no duplicate rows.

In [ ]:
def validate_kpi_table(df_kpi, name, expected_rows=None):
    print(f'\n===== {name} =====')
    print(f'  Shape      : {df_kpi.shape}')
    if expected_rows:
        row_ok = len(df_kpi) == expected_rows
        print(f'  Row check  : {"PASS" if row_ok else "FAIL"} (expected {expected_rows}, got {len(df_kpi)})')
    print(f'  Duplicates : {df_kpi.duplicated().sum()}')
    nulls = df_kpi.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        print(f'  Nulls      :')
        print(f'{nulls.to_string()}')
    else:
        print(f'  Nulls      : None')
    print(f'  Dtypes     :')
    print(df_kpi.dtypes.to_string())

validate_kpi_table(kpi_tournament, 'KPI TABLE 1 — Tournament', expected_rows=20)
validate_kpi_table(kpi_team,       'KPI TABLE 2 — Team Performance')
validate_kpi_table(kpi_player,     'KPI TABLE 3 — Player Stats')
validate_kpi_table(kpi_match,      'KPI TABLE 4 — Match Analytics', expected_rows=836)

---
## 7. KPI Sanity Checks

Cross-check key numbers against known facts to confirm accuracy.

In [ ]:
print('===== SANITY CHECKS =====')
print()

# Check 1: Total goals across all tournaments
total_goals_tournament = kpi_tournament['goalsscored'].sum()
total_goals_match      = kpi_match['total_goals'].sum()
print(f'[1] Total goals (tournament table) : {total_goals_tournament}')
print(f'    Total goals (match table)       : {total_goals_match}')
print(f'    Match: {"PASS" if total_goals_tournament == total_goals_match else "FAIL"}')
print()

# Check 2: Brazil is the top scorer (221 goals)
top_scorer = kpi_team.loc[kpi_team['goals_scored'].idxmax()]
print(f'[2] Top scoring team : {top_scorer["team"]} ({top_scorer["goals_scored"]} goals)')
print(f'    Match: {"PASS" if top_scorer["team"] == "Brazil" else "FAIL"}')
print()

# Check 3: Total matches across editions
total_matches_tournament = kpi_tournament['matchesplayed'].sum()
total_matches_match      = len(kpi_match)
print(f'[3] Total matches (tournament table): {total_matches_tournament}')
print(f'    Total matches (match table)      : {total_matches_match}')
print(f'    Match: {"PASS" if total_matches_tournament == total_matches_match else "FAIL"}')
print()

# Check 4: Home win rate ~57%
home_win_rate = kpi_match['is_home_win'].mean() * 100
print(f'[4] Home win rate: {home_win_rate:.1f}% (expected ~57.3%)')
print(f'    Match: {"PASS" if 55 < home_win_rate < 60 else "FAIL"}')
print()

# Check 5: 5 editions where host nation won
host_wins = kpi_tournament['host_nation_won'].sum()
print(f'[5] Host nation wins: {host_wins} (expected 5)')
print(f'    Match: {"PASS" if host_wins == 5 else "FAIL"}')
print()

# Check 6: Player with most goals is Ronaldo or Klose (both 16)
top_scorer_p = kpi_player.iloc[0]
print(f'[6] Top goal scorer: {top_scorer_p["player_name"]} ({int(top_scorer_p["goals"])} goals)')
print(f'    Match: {"PASS" if top_scorer_p["goals"] >= 15 else "FAIL"}')

print()
print('All checks complete.')

---
## 8. Export All KPI Tables to `data/processed/`

In [ ]:
exports = {
    '../data/processed/kpi_tournament.csv' : kpi_tournament,
    '../data/processed/kpi_team.csv'       : kpi_team,
    '../data/processed/kpi_player.csv'     : kpi_player,
    '../data/processed/kpi_match.csv'      : kpi_match,
}

print('Exporting KPI tables...')
print()
for path, table in exports.items():
    table.to_csv(path, index=False)
    size_kb = os.path.getsize(path) / 1024
    print(f'  {os.path.basename(path):<28} {table.shape[0]:>6} rows  x  {table.shape[1]:>2} cols  ({size_kb:.1f} KB)')

print()
print('All files saved to data/processed/')
print('Ready to connect in Tableau Public.')

---
## 9. Tableau Connection Guide

How to load and use each file in Tableau Public.

In [ ]:
guide = pd.DataFrame({
    'File': [
        'kpi_tournament.csv',
        'kpi_team.csv',
        'kpi_player.csv',
        'kpi_match.csv'
    ],
    'Grain': [
        '1 row per edition',
        '1 row per team',
        '1 row per player',
        '1 row per match'
    ],
    'Rows': [20, 83, 7638, 836],
    'Suggested Dashboard Use': [
        'Line chart: avg_goals_per_match by year | Bar: tournament_attendance by year | KPI tiles: top stats',
        'Bar chart: top teams by wins/goals | Scatter: win_rate vs goals_per_match | Filter by team',
        'Bar: top goal scorers | Table: player lookup | Filter by team/tournament',
        'Map: stadium locations | Scatter: attendance vs total_goals | Filter by year/stage/win_type'
    ],
    'Key Filter Fields': [
        'year, era, host_nation_won',
        'team, titles_won, tournament_appearances',
        'team, tournaments_attended, goals',
        'year, stage, is_knockout, win_type, era'
    ]
})
pd.set_option('display.max_colwidth', 90)
print(guide.to_string(index=False))

---
## 10. Final Pipeline Summary

Complete record of the full 5-notebook pipeline — for the Project Report.

In [ ]:
pipeline = pd.DataFrame({
    'Notebook': [
        '01_extraction.ipynb',
        '02_cleaning.ipynb',
        '03_eda.ipynb',
        '04_statistical_analysis.ipynb',
        '05_final_load_prep.ipynb'
    ],
    'Input': [
        'data/raw/ (3 CSVs)',
        'data/raw/ (3 CSVs)',
        'data/processed/world_cup_master.csv',
        'data/processed/world_cup_master.csv',
        'data/processed/world_cup_master.csv'
    ],
    'Output': [
        'Documented audit, data dictionary',
        'data/processed/world_cup_master.csv',
        '11 EDA visualisations + insights',
        '9 statistical tests + results log',
        '4 Tableau-ready KPI CSVs'
    ],
    'Key Action': [
        'Load raw, audit, validate requirements, document issues',
        'Drop blanks/dupes, merge 3 tables, clean 14 issues, export master',
        'Visualise distributions, trends, comparisons across 7 themes',
        '9 hypothesis tests: t-test, ANOVA, regression, correlation, binomial',
        'Compute 4 KPI tables, validate, export for Tableau'
    ]
})
pd.set_option('display.max_colwidth', 70)
print('===== END-TO-END PIPELINE SUMMARY =====')
print(pipeline.to_string(index=False))

print()
print('===== FINAL DATA OUTPUTS =====')
outputs = [
    ('world_cup_master.csv', '37,048 rows', '37 cols', 'Single clean analytical dataset'),
    ('kpi_tournament.csv',   '20 rows',     '14 cols', 'Tournament-level KPIs'),
    ('kpi_team.csv',         '83 rows',     '12 cols', 'All-time team performance'),
    ('kpi_player.csv',       '7,638 rows',  '13 cols', 'All-time player stats'),
    ('kpi_match.csv',        '836 rows',    '30 cols', 'Match-level analytics with flags'),
]
for fname, rows, cols, desc in outputs:
    print(f'  {fname:<30} {rows:<15} {cols:<10} {desc}')

print()
print('Next step: Open Tableau Public, connect to data/processed/, build the dashboard.')